# Identifying AI Image — Colab workbench

Runs the evidence-fusion MVP (provenance → watermarks → forensics → verdict) in Colab.

**Setup options** (pick one in the cell below):
- **A. Zip upload** — upload `ai-image-id-mvp.zip` when prompted.
- **B. GitHub** — once the repo is pushed, replace the URL and use the pip line instead.

In [1]:
# System dependency (exiftool) — required for metadata extraction
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!exiftool -ver

12.40


In [2]:
# --- Option A: upload the zip ---
#from google.colab import files
#uploaded = files.upload()  # select ai-image-id-mvp.zip
#!unzip -oq ai-image-id-mvp.zip
#%pip install -q -e ./ai-image-id-mvp

# --- Option B: from GitHub (uncomment once pushed) ---
!git clone -q https://github.com/Waranika/AI-image-Checkers.git
%cd AI-image-Checkers
%pip install -q -e .

/content/AI-image-Checkers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 3.0 MB/s eta 0:00:00
  Building editable for ai-image-id (pyproject.toml) ... done


In [3]:
# Sanity check
from ai_image_id.main import analyze_image
print("package OK")

package OK


In [4]:
!pwd
!git log --oneline -3
!grep -c "no AI-indicative" ai_image_id/fusion.py

/content/AI-image-Checkers
672b6a1 (HEAD -> main, origin/main, origin/HEAD) fixed bugs
1ca9efe Add CI badge to README
f9d7bc3 Restructure: flat layout, phase 2 modules, notebooks, CI
1


## Analyze any image

Upload an image and run the full pipeline. Try a Firefly/DALL·E export (C2PA), a
Meta AI image (IPTC tag), a raw SD/SDXL output (invisible watermark), and a phone photo.

In [5]:
# Batch analysis: clone the C2PA test corpus and run the whole folder
!git clone -q --depth 1 https://github.com/c2pa-org/public-testfiles.git

from pathlib import Path
from ai_image_id.main import analyze_image

folder = Path("public-testfiles/legacy/1.4/image/jpeg")

print(f"{'file':<46} {'verdict':<13} {'c2pa':<6} {'valid':<6} notes")
for p in sorted(folder.glob("*.jpg")):
    try:
        r = analyze_image(p)
        c = r.evidence.provenance
        note = r.notes[0][:60] if r.notes else ""
        print(f"{p.name:<46} {r.ai_verdict.value:<13} {str(c.c2pa_present):<6} {str(c.c2pa_valid):<6} {note}")
    except Exception as e:
        print(f"{p.name:<46} ERROR: {type(e).__name__}: {e}")

file                                           verdict       c2pa   valid  notes
adobe-20220124-A.jpg                           inconclusive  False  None   no AI-indicative signal found
adobe-20220124-C.jpg                           inconclusive  True   True   valid C2PA manifest from non-AI tool (make_test_images/0.16.
adobe-20220124-CA.jpg                          inconclusive  True   True   valid C2PA manifest from non-AI tool (make_test_images/0.16.
adobe-20220124-CACA.jpg                        inconclusive  True   True   valid C2PA manifest from non-AI tool (make_test_images/0.16.
adobe-20220124-CACAICAICICA.jpg                inconclusive  True   True   valid C2PA manifest from non-AI tool (make_test_images/0.16.
adobe-20220124-CAI.jpg                         inconclusive  True   True   valid C2PA manifest from non-AI tool (make_test_images/0.16.
adobe-20220124-CAIAIIICAICIICAIICICA.jpg       inconclusive  True   True   valid C2PA manifest from non-AI tool (make_test_images/0.16

## Demo: watermark round-trip

Embeds the SDXL 48-bit payload into a synthetic image, then verifies the pipeline
flags it as `verified`. Also shows fragility: JPEG-recompress the marked image and
watch bit accuracy drop — absence of a watermark is non-evidence.

In [6]:
import numpy as np
from PIL import Image
from ai_image_id.watermark import dwt_dct, SDXL_BITS

rng = np.random.default_rng(0)
y, x = np.mgrid[0:512, 0:512]
base = np.stack([120+60*np.sin(x/97), 100+50*np.cos(y/83), 90+40*np.sin((x+y)/71)], axis=-1)
clean = np.clip(base + rng.normal(0, 6, base.shape), 0, 255).astype(np.uint8)

marked = dwt_dct.embed(clean, SDXL_BITS)
Image.fromarray(marked).save("marked.png")            # lossless
Image.fromarray(marked).save("marked_q70.jpg", quality=70)  # lossy attack

for f in ["marked.png", "marked_q70.jpg"]:
    r = analyze_image(f)
    wm = next(w for w in r.evidence.watermarks if w.scheme == "dwtDct")
    print(f"{f}: verdict={r.ai_verdict.value}, bit_accuracy={wm.bit_accuracy}")

marked.png: verdict=verified, bit_accuracy=0.979
marked_q70.jpg: verdict=inconclusive, bit_accuracy=0.5


In [10]:
from google.colab import files
from ai_image_id.main import analyze_image
import json

up = files.upload()   # select one or several generated images
for name in up:
    r = analyze_image(name)
    print(f"\n=== {name} ===")
    print("verdict:", r.ai_verdict.value, r.confidence)
    p = r.evidence.provenance
    print("c2pa:", p.c2pa_present, p.c2pa_valid, "| generator:", p.c2pa_generator)
    print("iptc:", p.iptc_digital_source_type, "| hits:", p.ai_metadata_hits)
    print("notes:", r.notes)

Saving Screenshot 2026-07-06 at 23.58.56.png to Screenshot 2026-07-06 at 23.58.56.png

=== Screenshot 2026-07-06 at 23.58.56.png ===
verdict: inconclusive 0.5
c2pa: False None | generator: None
iptc: None | hits: []
notes: ['no AI-indicative signal found', 'absence of watermarks/metadata is non-evidence, not proof of human origin']


## Phase 2 workspace (learned detector)

Next steps, per the implementation plan — switch runtime to **GPU** for step 2:
1. Data prep: GenImage subset with matched JPEG-Q and resolution distributions
2. Precompute frozen DINOv2/CLIP embeddings (one pass, GPU)
3. Train attention-pooling head / linear probe (minutes, CPU-fine)
4. Temperature-scale calibration + cross-generator eval table
5. Export ONNX, add `detector` evidence to the fusion engine

In [7]:
# Phase 2 scaffold lands here — e.g.:
# %pip install -q torch timm
# import torch; torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')